# 性能工程——从采集到决策的完整闭环

> **性能工程**（建议 ~1.5h，含实操）。产物 ③：性能前后对比报告与决策结论。前置要求：03.02 节完成（算子正确性已验收）。素材：`src/perf_collect.py` + 基线。

> **本节新增知识只有 3 件（也是学完要记住的全部）**：① **计算密度**：gelu 每搬运 1 字节执行的指令数是 xlog1py 的 4 倍；② **四步流水分析法**：优化策略有系统路由，gelu 走 Step 1+4；③ **绝对时间标准**：验证看 L/M 档的绝对时间节省，防"大优化小劣化"的跷跷板。其余全部是基础课程 02.02 三原则的延续，只是换了算子，假设要重新检验。

> **写给第一次接触算子的读者**：算子代码由 AI 团队编写与修复，指挥官角色看得懂结果、做得了决定就够。

**本模块目标**：
1. 采集 gelu_approx 多用例基线并按规模档归因；
2. 建立**计算密度**概念（2.2 由你先推导），再用自己采集的数据检验访存受限假设是否仍成立；
3. 用**四步流水分析法**路由优化策略，实施一项优化并完成双回归与达标判定。

## 1. 26 万元素和 17 元素，耗时一模一样

真机实测数据（本课程开发时的真实采集）：

| 用例 | 元素数 | 耗时 |
|---|---|---|
| A | 260,000 | 0.160 ms |
| B | 17 | 0.160 ms |

耗时完全不随计算量变化：瓶颈在**launch 固定开销**（~0.16ms），优化 kernel（算子核心代码）计算一个字节都省不下来。这就是基础课程 02.02 数据先于观点的实操版：基础课程看 xlog1py 的实录数字，这里亲手采 gelu 的；分档读法也原样适用（L 看带宽、S 看时延地板）。
运行下面的 cell 可视化 launch 地板：规模横跨 5 个数量级，耗时几乎恒定在 ~0.16ms：


In [ ]:
# "26万元素和17元素一样快"——launch 地板可视化
import json, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 加载基线（优先用户采集，退回内置）
paths = ['results/perf_baseline.json', 'src/perf_baseline.json']
data = None
for p in paths:
    if os.path.exists(p):
        data = json.load(open(p)); break
if data is None:
    data = {"rows": [
        {"case": "L_gelu_2d", "numel": 2555392, "ms": 0.1629, "gbps": 125.5},
        {"case": "L_gelu_1d", "numel": 71934, "ms": 0.1661, "gbps": 3.5},
        {"case": "M_gelu_3d", "numel": 25004304, "ms": 0.2616, "gbps": 764.8},
        {"case": "S_gelu_small", "numel": 9, "ms": 0.1600, "gbps": 0.0},
        {"case": "S_gelu_2d", "numel": 1060, "ms": 0.1638, "gbps": 0.1},
        {"case": "A_gelu_align", "numel": 2617218, "ms": 0.1711, "gbps": 122.4},
    ]}

rows = sorted(data['rows'], key=lambda r: r['numel'])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 左图：规模 vs 耗时（log-log）
sizes = [r['numel'] for r in rows]; times = [r['ms'] for r in rows]
ax1.loglog(sizes, times, 'o-', color='#C7000B', markersize=8)
ax1.axhline(y=0.16, color='gray', ls='--', alpha=0.5, label='launch 地板 ~0.16ms')
ax1.set_xlabel('元素数'); ax1.set_ylabel('耗时 (ms)')
ax1.set_title('规模 vs 耗时（launch 地板）', fontsize=10)
ax1.legend(fontsize=8)
for r in rows:
    if r['numel'] < 100000:
        ax1.annotate(r['case'], (r['numel'], r['ms']), fontsize=6, ha='right')

# 右图：有效带宽
cases = [r['case'][:12] for r in rows]; bw = [r['gbps'] for r in rows]
bars = ax2.bar(cases, bw, color=['#2A6EBB' if b > 100 else '#B8860B' for b in bw])
ax2.set_ylabel('有效带宽 (GB/s)'); ax2.set_title('有效带宽分布', fontsize=10)
ax2.tick_params(axis='x', rotation=45, labelsize=7)
ax2.axhline(y=1600, color='red', ls=':', alpha=0.5, label='HBM 峰值 ~1.6TB/s')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('launch_floor.png', dpi=100, bbox_inches='tight')
plt.show()
print("→ 小规模耗时恒定 ~0.16ms（launch 地板）；最大档 765 GB/s ≈ 48% 峰值")

## 2. 从 xlog1py 到 gelu_approx：访存受限还成立吗？

### 2.1 回顾：xlog1py 为什么是访存受限

基础课程 02.02（自学：阶段二 02.02）的结论链：xlog1py 每处理 1 个元素搬运 12 字节（x 4B 读 + y 4B 读 + z 4B 写），只执行约 3 条向量指令，**大部分时间在等数据搬运**。所以优化看有效带宽 vs HBM 峰值（实测 64%），方向是让搬运更满。

### 2.2 新概念：计算密度

**计算密度 = 每搬运 1 字节执行的向量指令条数**（条/字节），衡量等数据的时间里计算占多重。心算填空：

- xlog1py：约 3 条指令 / 12 字节 = ______ 条/字节
- gelu_approx：约 8 条指令 / 8 字节（x 4B 读 + z 4B 写）= ______ 条/字节
- 密度比：______ 倍

<details><summary>核对</summary>

0.25 · 1.0 · **4 倍**。gelu 每搬运 1 字节要多执行 4 倍的指令。

</details>

可视化核对与后果链（64%→48% 的因果，3.2 用你自己采集的数据验证）：

<img src="./images/03.03_compute_density.png" alt="计算密度对比：xlog1py 0.25 vs gelu 1.0 条/字节，密度 4 倍的后果链" width="700px">

### 2.3 问题：假设还成立吗？

基础课程 02.02 结尾留过一个问题："如果一个算子每搬运 1 字节要执行 4 倍的指令，访存受限还成立吗？"现在有了密度数据，可以正式回答。如果计算密度提高 4 倍，计算时间可忽略这个访存受限前提可能不再成立：搬运等待期间，计算可能开始排队，带宽利用率反而下降。

**这是猜测，还是事实？** 不用猜，第 3 节将用自己采集的数据回答。带着这个问题，进入 Lab C。


## 3. Lab C：gelu_approx 性能闭环（50min）

> **执行位置**：以下命令在课程仓库 `tutorials/CANNBot/03_advanced_path/` 目录下执行（素材在 `src/` 子目录）。

### 3.1 采集基线（15min）

```bash
cd src && python3 perf_collect.py --out results/perf_baseline.json
```

> **预期现象**：6 个用例逐行打印。M 档用例带宽最高（~765 GB/s），S 档全部 ~0.16ms 恒定。

### 3.2 分档归因（10min）

**真机基线对照**（Ascend 910B3，2026-10 实测，与你的采集结果对照）：

| 用例 | 规模 | 耗时 | 有效带宽 |
|---|---|---|---|
| L_gelu_2d | 256 万 | 0.163 ms | 126 GB/s |
| M_gelu_3d | 2500 万 | 0.262 ms | **765 GB/s** |
| L_gelu_1d | 7.2 万 | 0.166 ms | 3.5 GB/s |
| S_gelu_small | 9 | 0.160 ms | ≈0 |
| S_gelu_2d | 1,060 | 0.164 ms | 0.1 GB/s |
| A_gelu_align | 262 万 | 0.171 ms | 122 GB/s |

对照数据回答三个问题（这是产物 ③ 的归因部分）：

1. M 档 765 GB/s ≈ HBM 峰值的 **48%**，阶段二 xlog1py 同规模档是 64%。gelu 的计算链更长，带宽反而更低，为什么？（提示：9.2.3 的问题）
2. S 档全部 ~0.16ms 恒定，瓶颈是什么？对应分档读法的哪一条？
3. 中大规模档 126 GB/s 未饱和，给出一项可落地的优化方向。

<details><summary>参考归因（先自己答，再对照）</summary>

1. **计算密度提高 4 倍后，计算时间开始占用流水线**，搬运等计算，带宽利用率下降。纯访存受限假设部分失效，优化方向从让搬运更满扩展到让计算与搬运重叠；
2. **launch 地板**（~0.16ms 固定开销），S 档看时延恒定性；
3. tile 大小调优 / 双缓冲流水（CopyIn-Compute-CopyOut 重叠）/ 合并 Mul 步骤降低计算密度，任选其一，3.3 落地。

</details>

**三档策略对照**（归因之后、动手之前的速查）：

| 档位 | 瓶颈 | 代表数据（真机） | 策略方向 | 验证标准 |
|---|---|---|---|---|
| L 档（百万级） | 访存为主、未饱和 | 126 GB/s | 流量削减：tile 调优、紧凑输入 | 绝对时间节省 |
| M 档（千万级） | 计算开始占用流水线 | 765 GB/s ≈ 48% 峰值（xlog1py 同档 64%） | 计算搬运重叠：双缓冲、降密度 | 绝对时间节省（价值主体） |
| S 档（微小） | launch 地板 | ~0.16ms 恒定 | 算子层之外：融合、减调用 | 时延恒定性，kernel 层无空间 |

### 3.3 优化实施（15min）

**策略从哪来？** 不是随意挑一项：基础课程 02.02 说"与瓶颈匹配的一轮优化"，`ascendc-perf-optimize` Skill 把这句话系统化为**四步流水分析法**（逐层路由，每步回修 Step 1 的 Tiling）：

| 步 | 方法 | gelu_approx |
|---|---|---|
| 1 | **Tiling 理论建模**（切分 + Buffer 规划） | ✔ |
| 2 | 通算演算优化（仅通信类算子） | ✘ 自动跳过 |
| 3 | 核间流水优化（仅多核同步算子） | ✘ 自动跳过 |
| 4 | **单核流水优化**：bound 判定 + 核内策略 | ✔ 主战场 |

**Step 4 的 bound 判定路由**（对照你 3.2 的归因）：

<img src="./images/03.03_bound_routing.png" alt="Step 4 单核流水 bound 判定路由决策树：Scalar/访存/Vec/无 Bound 四分支，gelu 高亮 Vec Bound" width="700px">

gelu 计算密度 4 倍 → Vector 流水开始占用 → **Vec Bound 倾向**。落地项任选其一：

- **tile 大小调优**（Step 1 回修）：扫描 tileElems 找最优（UB 利用率最大化）；
- **双缓冲流水**（Step 4 核内）：确认 CopyIn-Compute-CopyOut 重叠是否生效；
- **计算密度降低**（Step 4 vec 策略）：合并 Mul 步骤减少中间变量，降低计算-访存比。

方法论一句话：Step 2/3/4 的结论回修 Step 1 的 Tiling；非适用步骤自动跳过（gelu 跳过 2/3 正是例证）。

### 3.4 双回归与达标判定（10min）

```bash
# 性能回归
python3 perf_collect.py --out results/perf_optimized.json
# 精度回归
cd ../src/gelu_approx_naive/tests && python3 -m pytest test_gelu_approx_blackbox.py -q
```

**达标判定分三档**（一句话版）：**PASS** 交付 / **PASS WITH NOTES** 基本达标附边界条件 / **FAIL** 回 9.3.3 迭代。

**验收标准**：价值主体看 **L/M 档的绝对时间节省**，这两档占总执行时间 95%+；把所有用例取平均反而会掩盖真实收益（小用例的固定开销会拉低平均数）。警惕**跷跷板**：大档优化、小档劣化，必须加 S 档快速路径消除。逐用例回归是红线。


## 4. 要点回顾

**本章数字锚点**：3 原则承接 · 1 新概念 · 4 步分析法 · 3 种判定

- 三原则实操闭环：**采集 → 分档归因 → 优化 → 双回归**，只是换算子重验假设；
- **计算密度**（新）：gelu 1.0 vs xlog1py 0.25 条/字节，M 档 48%（vs 64%）说明计算开始占用流水线，纯搬运优化已不够；S 档 launch 地板照旧；
- **四步流水分析法**（新）：gelu 走 Step 1+4，bound 路由到 Vec；
- **验收标准**（新）：L/M 档绝对时间是价值主体，防跷跷板；逐用例回归是红线。

**下一站**：[03.04 · 反哺与结业](./03.04_skill_and_assessment.ipynb)将把今天的精度预算经验和性能发现，沉淀进 Skill。

## 5. 带走物

| 类别 | 内容 |
|---|---|
| **一句话** | 密度变了，假设要重验：gelu 4 倍密度打破了 xlog1py 的访存假设 |
| **工具** | 计算密度分析 + 四步流水分析法（gelu 走 Step 1+4） |
| **产物** | 性能对比报告（逐用例 + 计算密度分析 + 双回归） |
| **模板** | 三档策略对照（L 流量 / M 重叠 / S 融合）+ 绝对时间标准 |
| **下一步** | [03.04](./03.04_skill_and_assessment.ipynb)：把经验沉淀进 Skill |

## 6. 章末检查点

1. **（单选）** gelu_approx 的计算密度约是 xlog1py 的几倍？
   - A. 相同  B. 4 倍（8 条指令/8 字节 vs 3 条/12 字节）  C. 8 倍  D. 无法比较

2. **（单选）** M 档带宽 765 GB/s（48% 峰值），未饱和的原因？
   - A. 数据量不够  B. 计算密度提高，计算时间开始占用流水线  C. NPU 有故障  D. 带宽测量误差

3. **（简答）** 你的优化让 M 档有提升但 L 档不变，这说明了什么？

<details><summary>参考答案</summary>

1. **B** 2. **B**
3. L 档规模太小（256 万元素），launch 开销仍占主导（0.163ms 中 launch 占 ~0.16ms，计算仅 ~0.003ms），launch 地板限制了 L 档的天花板；M 档规模够大，计算/搬运优化才能生效。
</details>

## 7. 产物自查表（交付 ③）

| 项 | 标准 | 自查 |
|---|---|---|
| 基线表 | ≥6 用例，含 L/M/S 档 | ☐ |
| 归因结论 | 含计算密度分析（新） | ☐ |
| 优化实施 | 一项落地（diff 可查，经四步法路由） | ☐ |
| 双回归 | 逐用例性能 + 全量精度绿（三档判定明确） | ☐ |
| 决策结论 | 含何时不该优化 | ☐ |

## 8. 课后练习

> 练习答案：[`answer/03.03_answer.txt`](./answer/03.03_answer.txt)

1. **（单选题）** gelu 计算密度 1.0 条/字节（xlog1py 为 0.25），M 档带宽却从 64% 降到 48%，最合理的解释？
   - A. gelu 的 kernel 写错了
   - B. 计算时间开始占用流水线，搬运等计算，带宽利用率下降（访存受限假设部分失效）
   - C. NPU 检测到长指令链自动降频
   - D. 带宽测量工具对 8 步链不准

2. **（单选题）** 四步流水分析法中 gelu 跳过 Step 2（通算演算）与 Step 3（核间流水）的依据？
   - A. 这两步难度太高，留给专家
   - B. 三层覆盖：非通信算子/无跨核依赖，非适用步骤自动跳过
   - C. 这两步只对 fp16 生效
   - D. 讲师指定的捷径

3. **（多选题）** 优化后 L 档加速 1.3×、S 档退化到 0.8×，正确的处理方式包括？
   - A. 按全 case 几何平均发布（1.1×，收益为正）
   - B. 识别为跷跷板效应，加 S 档快速路径消除退化
   - C. 验收看绝对时间节省（L/M 档占总执行时间 95%+），但逐用例回归是红线
   - D. 直接合入，S 档 0.2 的退化在容差内

4. **（编程实践·必做）** **第二迭代**：对你本节 Lab C 的优化（或 03.01 练习交付的 fp16 gelu）换一条四步法路由的策略再迭代一轮：若上轮走 Step 1（tile 调优），本轮走 Step 4（双缓冲流水或计算密度降低）。完成后用绝对时间节省标准重写结论，并与第一轮对比：哪轮收益大、为什么？

5. **（编程实践·选做）** **六维性能复盘**：为 xlog1py（阶段二 02.02）与 gelu（本节）写一张对照复盘：计算密度 / M 档带宽 / launch 地板 / 路由的策略 / 实测收益 / 停止条件。指出密度 4 倍这一个变量如何改变了三行以上的策略选择。
